# Week 9 In-Class Exercise: Dimensionality Reduction on Handwritten Digits

In the textbook (Chapter 8), we studied how to reduce the number of features in a dataset while keeping most of the information: **PCA** (projecting onto the directions of greatest variance), choosing the number of components from the **explained variance ratio**, **PCA for compression** (and reconstruction), **Kernel PCA** for nonlinear structure, and **manifold learning** methods like **LLE** and **t-SNE** for visualization.

In this exercise, you'll apply those **same concepts** to a different dataset:

> **Digits** — 1,797 images of handwritten digits (0-9), each an 8x8 grayscale image flattened into **64 features**. It ships with scikit-learn, so there is nothing to download.

64 features is small enough to run instantly but large enough that dimensionality reduction genuinely helps. Every technique from the chapter transfers directly — your job is to adapt the code, not invent new methods.

You'll follow the same workflow as Chapter 8:
1. Load and look at the data
2. Fit **PCA** and read the **explained variance ratio**
3. Pick the number of components that keeps **95%** of the variance
4. **Compress and reconstruct** the images to see what PCA throws away
5. Visualize the digits in **2D** with PCA and with **t-SNE**
6. Use PCA as a **preprocessing step** before a classifier

### Useful API References

| Task | Documentation |
|------|---------------|
| Load data | [sklearn.datasets.load_digits](https://scikit-learn.org/stable/modules/generated/sklearn.datasets.load_digits.html) |
| PCA | [decomposition.PCA](https://scikit-learn.org/stable/modules/generated/sklearn.decomposition.PCA.html) |
| Kernel PCA | [decomposition.KernelPCA](https://scikit-learn.org/stable/modules/generated/sklearn.decomposition.KernelPCA.html) |
| t-SNE | [manifold.TSNE](https://scikit-learn.org/stable/modules/generated/sklearn.manifold.TSNE.html) |
| Scaling | [preprocessing.StandardScaler](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.StandardScaler.html) |
| Pipelines | [pipeline.make_pipeline](https://scikit-learn.org/stable/modules/generated/sklearn.pipeline.make_pipeline.html) |
| Classifier | [linear_model.LogisticRegression](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LogisticRegression.html) |
| Metrics | [accuracy_score](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.accuracy_score.html), [cross_val_score](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.cross_val_score.html) |

<table align="left">
  <td>
    <a href="https://colab.research.google.com/github/pjmcswee/IST707-Notebooks/blob/main/week9/week9_inclass_exercise.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>
  </td>
</table>

## Setup

Run this cell to import the libraries and set the plotting defaults.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

plt.rc('font', size=13)
plt.rc('axes', labelsize=13, titlesize=13)
plt.rc('legend', fontsize=11)
plt.rc('xtick', labelsize=10)
plt.rc('ytick', labelsize=10)

RANDOM_STATE = 42

## Step 1: Load and Look at the Data (Provided)

Each row is one 8x8 image flattened into 64 pixel-intensity values (0-16). The target is the digit (0-9). This cell is provided — just run it to load the data and preview a few digits.

In [ ]:
from sklearn.datasets import load_digits

digits = load_digits()
X = digits.data       # shape (1797, 64)
y = digits.target     # shape (1797,)

print('Data shape:', X.shape)
print('Number of features (pixels):', X.shape[1])
print('Classes:', np.unique(y))

# Preview the first 10 digits
fig, axes = plt.subplots(1, 10, figsize=(12, 2))
for i, ax in enumerate(axes):
    ax.imshow(X[i].reshape(8, 8), cmap='binary')
    ax.set_title(str(y[i]))
    ax.axis('off')
plt.show()

## Step 2: Fit PCA and Read the Explained Variance

PCA finds the orthogonal directions (principal components) along which the data varies the most. The first component captures the most variance, the second the next most, and so on. `explained_variance_ratio_` tells you the fraction of total variance each component captures.

**Your turn!** Fit a `PCA()` (with no `n_components`, so it keeps all 64) on `X`, then print the explained variance ratio of the **first 5** components, and the **total** variance they capture together. Also print the ratio of a *late* component (e.g. the 30th) so you can compare it against the first.

**Hint (from the chapter):**
```python
from sklearn.decomposition import PCA
pca = PCA(random_state=RANDOM_STATE)
pca.fit(X)
pca.explained_variance_ratio_[:5]      # first five
pca.explained_variance_ratio_[29]      # the 30th, for contrast
```

In [ ]:
# TODO: Fit PCA on X. Print the explained variance ratio of the first 5
#       components, their running total, and (for contrast) the 30th.
# Your code here:




**Question:** The first principal component captures far more variance than a later one (like the 30th). In plain terms, what does it mean for a direction in pixel-space to have "high variance," and why does that make it worth keeping?

*Your answer:*



## Step 3: How Many Components for 95% of the Variance?

Instead of guessing the number of components, we can plot the **cumulative** explained variance and pick the smallest number of components that reaches a target (here, 95%).

**Your turn!**
1. Compute the cumulative sum of `pca.explained_variance_ratio_` and plot it (x = number of components, y = cumulative variance). Draw a horizontal line at 0.95.
2. Find and print the smallest number of components `d` that reaches 95%.

**Hint (from the chapter):**
```python
cumsum = np.cumsum(pca.explained_variance_ratio_)
d = np.argmax(cumsum >= 0.95) + 1
```

In [ ]:
# TODO: Plot cumulative explained variance and print the number of components
#       needed to reach 95%.
# Your code here:




**Question:** You started with 64 features. How many did you need to keep 95% of the variance? What does that compression ratio tell you about how much redundancy there is between the 64 pixels of a digit image?

*Your answer:*



## Step 4: Compress and Reconstruct

PCA is lossy compression: project down to `d` dimensions, then `inverse_transform` back to 64 to see what survived. The reconstruction is blurrier because the discarded components held the fine detail.

**Your turn!**
1. Fit `PCA(n_components=0.95)` on `X` (passing a float tells PCA to keep enough components for 95% variance automatically).
2. Transform `X` down, then `inverse_transform` back to 64 dimensions.
3. Display the first 5 **original** digits in a top row and their **reconstructions** in a bottom row.

**Hint (from the chapter):**
```python
pca = PCA(n_components=0.95, random_state=RANDOM_STATE)
X_reduced = pca.fit_transform(X)
X_recovered = pca.inverse_transform(X_reduced)
# then imshow X[i].reshape(8, 8) and X_recovered[i].reshape(8, 8)
```

In [ ]:
# TODO: Compress to 95% variance, reconstruct, and show original vs
#       reconstructed digits side by side.
# Your code here:




**Question:** The reconstructed digits are still clearly readable even though you threw away most of the components. What kind of information did PCA discard — and why is the number still recognizable?

*Your answer:*



## Step 5: Visualize in 2D — PCA vs t-SNE

To *see* 64-dimensional data, we squeeze it down to 2D and make a scatter plot colored by digit. PCA is a fast **linear** projection; **t-SNE** is a slower **nonlinear** method that tries hard to keep similar points close together, usually giving cleaner clusters.

**Your turn!**
1. Reduce `X` to 2D with `PCA(n_components=2)` and scatter-plot it, colored by `y`.
2. Reduce `X` to 2D with `TSNE(n_components=2)` and scatter-plot that, colored by `y`.
3. Put the two plots side by side so you can compare the clusters.

**Hint (from the chapter):**
```python
from sklearn.manifold import TSNE
X_pca = PCA(n_components=2, random_state=RANDOM_STATE).fit_transform(X)
X_tsne = TSNE(n_components=2, init='pca', learning_rate='auto',
              random_state=RANDOM_STATE).fit_transform(X)
plt.scatter(X_pca[:, 0], X_pca[:, 1], c=y, cmap='tab10', s=10)
```

In [ ]:
# TODO: Make a 2D PCA scatter and a 2D t-SNE scatter, both colored by digit,
#       side by side. Add a colorbar or legend so the digits are identifiable.
# Your code here:




**Question:** Compare the two plots. Which one separates the 10 digit classes more cleanly? Give one reason t-SNE can pull apart clusters that PCA leaves overlapping.

*Your answer:*



## Step 6: PCA as Preprocessing for a Classifier

Dimensionality reduction isn't only for pictures — it can speed up (and sometimes improve) a model by feeding it fewer, denser features. Here we compare a `LogisticRegression` trained on all 64 features against one trained on the PCA-reduced features.

**Your turn!**
1. Measure the **5-fold cross-validation accuracy** of `LogisticRegression(max_iter=5000)` on the raw `X`.
2. Build a pipeline `make_pipeline(PCA(n_components=0.95), LogisticRegression(max_iter=5000))` and measure its 5-fold CV accuracy on `X`.
3. Print both accuracies and how many features PCA kept.

**Hint (from the chapter):**
```python
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score
scores = cross_val_score(model, X, y, cv=5)
```

In [ ]:
# TODO: Compare CV accuracy of LogisticRegression on raw X vs a
#       PCA(n_components=0.95) + LogisticRegression pipeline. Print both and
#       the kept dim.
# Your code here:




**Question:** Did reducing from 64 features to the PCA components change the accuracy much? When PCA cuts the feature count roughly in half with little or no accuracy loss, what have you gained?

*Your answer:*



## Reflection Questions

Answer each question in the cell below it (1-3 sentences each).

**Q1:** PCA keeps the directions of **greatest variance**. Give one example of a situation where the direction of greatest variance is **not** the most useful direction for a prediction task — i.e., where PCA might discard something important.

*Your answer:*



**Q2:** You used t-SNE to visualize the digits, but we did **not** use it as preprocessing before the classifier the way we used PCA. Based on the chapter, give one reason t-SNE is well suited to visualization but a poor choice for transforming features that new, unseen data must pass through.

*Your answer:*



**Q3:** The chapter introduced **Kernel PCA** for data that lies on a curved (nonlinear) manifold, like the Swiss roll. For the digits data, standard linear PCA already worked well. What does that suggest about the shape of the digit data in 64-dimensional space, compared to the Swiss roll?

*Your answer:*

